**Bank Management System**


In [19]:
from abc import ABC, abstractmethod


class Account(ABC):

    def __init__(self, account_number, account_name, balance):
        self.account_number = account_number  # Public attribute
        self._account_name = account_name  # Protected attribute
        self.__balance = balance  # Private attribute

    def show_account_number(self):
        return f"Account Number: {self.account_number}"

    def show_account_name(self):
        return f"Account Name: {self._account_name}"

    # Getter method for accessing the private attribute __balance
    def get_balance(self):
        return self.__balance

    # Update balance method used for deposits (adds to the existing balance)
    def update_balance(self, amount):
        if amount > 0:
            self.__balance += amount
            print(f"Deposited {amount}. {self._account_name}'s new balance: {self.__balance}")
        else:
            print("Update failed: Deposit amount must be greater than zero.")

    # Helper method for internal balance updates (used to set exact balance after withdrawals)
    def _set_balance_direct(self, amount):
        self.__balance = amount

    @abstractmethod
    def withdraw(self, amount):
        pass


class SavingsAccount(Account):

    def __init__(self, account_number, account_name, balance, interest_rate):
        super().__init__(account_number, account_name, balance)
        self.interest_rate = interest_rate

    def withdraw(self, amount):
        if amount < 0:
            print("Withdrawal amount cannot be negative.")
        elif amount <= self.get_balance():
            new_bal = self.get_balance() - amount
            # Using the direct setter method here because update_balance is exclusively for deposits
            self._set_balance_direct(new_bal)  
            print(f"Withdrew {amount}. New balance: {self.get_balance()}")
        else:
            print("Insufficient funds.")


class CurrentAccount(Account):

    def __init__(self, account_number, account_name, balance, overdraft_limit):
        super().__init__(account_number, account_name, balance)
        self.overdraft_limit = overdraft_limit

    def withdraw(self, amount):
        if amount < 0:
            print("Withdrawal amount cannot be negative.")
        elif amount <= self.get_balance() + self.overdraft_limit:
            new_bal = self.get_balance() - amount
            # Overdraft can cause a negative balance, which is handled properly by the direct setter
            self._set_balance_direct(new_bal)
            
            # Loan checking 
            if new_bal < 0:
                print(f"Withdrew {amount}. New balance: {self.get_balance()} (Overdraft used: {-new_bal})")
            else:
                print(f"Withdrew {amount}. New balance: {self.get_balance()}")
        else:
            print("Insufficient funds and Overdraft Limit Exceeded.")


# ---------------- Testing ----------------

savings_acc = SavingsAccount("SA123", "Alice", 1000, 0.05)

# 1. Update balance (Deposit) : 1000 + 500 = 1500
savings_acc.update_balance(500)
# Output: Deposited 500. Alice's new balance: 1500

# 2. Negative deposit attempt
savings_acc.update_balance(-200)
# Output: Update failed: Deposit amount must be greater than zero.

# 3. Withdraw Test : Deducts 200 from 1500, making it 1300
savings_acc.withdraw(200)

current_acc = CurrentAccount("CA456", "Bob", 500, 200)
current_acc.withdraw(700)

Deposited 500. Alice's new balance: 1500
Update failed: Deposit amount must be greater than zero.
Withdrew 200. New balance: 1300
Withdrew 700. New balance: -200 (Overdraft used: 200)
